In [61]:
import random
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json

In [62]:
seed        = 42
device      = torch.device("mps" if torch.mps.is_available() else "cpu")
emb_dim     = 64
hid_dim     = 128
n_layers    = 2
dropout     = 0.3
batch_size  = 64
n_epochs    = 20
clip        = 1.0
teacher_forcing_ratio = 0.5

random.seed(seed)
torch.manual_seed(seed)

In [63]:
pad, sos, eos, unk = 0, 1, 2, 3
with open("../datas/couplet/vocabs", "r") as f:
    vocab = f.read().split('\n')
print(len(vocab))
with open('../datas/couplet/char2id.json', 'r', encoding='utf-8') as f:
    tok2idx = json.load(f)
with open('../datas/couplet/id2char.json', 'r', encoding='utf-8') as f:
    idx2tok = json.load(f)
print(len(tok2idx))
print(len(idx2tok))
vocab_size = len(vocab)
# print(idx2tok.items())
# print(tok2idx.items())

9132
9132
9132


In [52]:
with open('../datas/couplet/train/out_id.json', 'r', encoding='utf-8') as f:
    train_out_id = json.load(f)
with open('../datas/couplet/train/in_id.json', 'r', encoding='utf-8') as f:
    train_in_id = json.load(f)
with open('../datas/couplet/test/out_id.json', 'r', encoding='utf-8') as f:
    test_out_id = json.load(f)
with open('../datas/couplet/test/in_id.json', 'r', encoding='utf-8') as f:
    test_in_id = json.load(f)
with open('../datas/couplet/train/true_len.json', 'r', encoding='utf-8') as f:
    train_true_len = json.load(f)
with open('../datas/couplet/test/true_len.json', 'r', encoding='utf-8') as f:
    test_true_len = json.load(f)
print(type(train_true_len))

<class 'list'>


In [53]:
max_len = max(len(sentence) for sentence in train_out_id)
max_len +=2
max_len

36

In [54]:
print(train_out_id[:10])
print(train_in_id[:10])
print(train_true_len[:10])
print(test_true_len[:10])

[[1, 901, 325, 459, 12, 12, 206, 43, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 23, 113, 115, 873, 20, 262, 4220, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 84, 67, 1251, 32, 178, 752, 25, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 4215, 4215, 98, 461, 349, 5, 4967, 4380, 362, 187, 5, 881, 3312, 362, 187, 5, 206, 1479, 365, 92, 1762, 289, 85, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 122, 147, 53, 76, 58, 81, 21, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 168, 92, 1018, 351, 5, 168, 351, 1018, 92, 5, 3231, 815, 49, 1099, 17, 1596, 68, 5, 972, 1167, 1563, 282, 5, 18, 154, 1304, 282, 2, 0, 0, 0, 0, 0], [1, 1711, 110, 301, 268, 306, 59, 335, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 503, 190, 2196, 1662, 192, 5, 29, 17, 75, 2125, 2411, 17, 63, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [64]:
class MyDataset(Dataset):
    def __init__(self, data, target, true_lens):

        self.data = torch.tensor(data,dtype=torch.long)
        self.target = torch.tensor(target,dtype=torch.long)
        self.true_lens = torch.tensor(true_lens,dtype=torch.long)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.target[idx],self.true_lens[idx]

In [65]:
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v    = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs, src_mask=None):
        """
        dec_hidden : (batch, dec_hid)
        enc_outputs: (batch, src_len, enc_hid*2)
        """
        src_len = enc_outputs.shape[1]
        dec_hidden = dec_hidden.unsqueeze(1).repeat(1, src_len, 1)   # (B, L, H)
        energy = torch.tanh(self.attn(torch.cat([dec_hidden, enc_outputs], dim=2)))
        attention = self.v(energy).squeeze(2)                          # (B, L)
        if src_mask is not None:
            attention = attention.masked_fill(src_mask == 0, -1e10)
        return F.softmax(attention, dim=1)

In [66]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, n_layers,
                          bidirectional=True, batch_first=True, dropout=dropout)
        self.fc  = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        """src: (batch, src_len)"""
        embedded = self.dropout(self.embedding(src))            # (B, L, E)
        outputs, hidden = self.rnn(embedded)                    # outputs: (B, L, H*2)
        # 取最后一层的前向+后向隐层，拼接后映射到 dec_hid_dim
        hidden = torch.tanh(self.fc(
            torch.cat([hidden[-2], hidden[-1]], dim=1)          # (B, H*2)
        ))                                                       # (B, dec_hid)
        return outputs, hidden


In [67]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad)
        self.rnn = nn.GRU(enc_hid_dim * 2 + emb_dim, dec_hid_dim,
                          batch_first=True)
        self.fc_out = nn.Linear(enc_hid_dim * 2 + dec_hid_dim + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt_token, hidden, enc_outputs, src_mask=None):
        """
        tgt_token : (batch,)
        hidden    : (1, batch, dec_hid)
        enc_outputs: (batch, src_len, enc_hid*2)
        """
        tgt_token = tgt_token.unsqueeze(1)                       # (B, 1)
        embedded  = self.dropout(self.embedding(tgt_token))      # (B, 1, E)

        attn_weights = self.attention(hidden.squeeze(0), enc_outputs, src_mask)
        attn_weights = attn_weights.unsqueeze(1)                 # (B, 1, L)
        context = torch.bmm(attn_weights, enc_outputs)           # (B, 1, H*2)

        rnn_input = torch.cat([embedded, context], dim=2)        # (B, 1, E+H*2)
        output, hidden = self.rnn(rnn_input, hidden)             # output: (B,1,dec_hid)

        pred = self.fc_out(
            torch.cat([output, context, embedded], dim=2)        # (B,1, dec_hid+H*2+E)
        ).squeeze(1)                                             # (B, vocab)
        return pred, hidden

In [68]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def make_src_mask(self, src):
        return (src != pad)                                      # (B, L)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src : (B, src_len)
        tgt : (B, tgt_len)
        """
        B, tgt_len = tgt.shape
        vocab_size  = self.decoder.fc_out.out_features

        outputs   = torch.zeros(B, tgt_len, vocab_size).to(self.device)
        src_mask  = self.make_src_mask(src)
        enc_out, hidden = self.encoder(src)

        dec_input = tgt[:, 0]                                    # <sos>
        hidden    = hidden.unsqueeze(0)                          # (1, B, H)

        for t in range(1, tgt_len):
            pred, hidden = self.decoder(dec_input, hidden, enc_out, src_mask)
            outputs[:, t] = pred
            use_teacher = random.random() < teacher_forcing_ratio
            top1 = pred.argmax(1)
            dec_input = tgt[:, t] if use_teacher else top1

        return outputs

    @torch.no_grad()
    def translate(self, src, max_len=max_len):
        """贪婪解码推理"""
        self.eval()
        src = src.to(self.device)
        src_mask = self.make_src_mask(src)
        enc_out, hidden = self.encoder(src)
        hidden = hidden.unsqueeze(0)

        dec_input = torch.tensor([sos], device=self.device)
        result = []
        for _ in range(max_len):
            pred, hidden = self.decoder(dec_input, hidden, enc_out, src_mask)
            token = pred.argmax(1)
            if token.item() == eos:
                break
            result.append(token.item())
            dec_input = token
        return result

In [69]:
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    total_loss = 0
    for src, tgt,_ in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio)        # (B, tgt_len, V)
        # 忽略 <sos> 位置
        output = output[:, 1:].reshape(-1, vocab_size)
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [70]:
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt,_ in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            output = output[:, 1:].reshape(-1, vocab_size)
            target = tgt[:, 1:].reshape(-1)
            total_loss += criterion(output, target).item()
    return total_loss / len(loader)

In [71]:
def main():
	print(f"Using device: {device}\n")

	# 数据
	X_train,y_train,X_test,y_test = train_in_id,train_out_id,test_in_id,test_out_id
	ture_lens_train,ture_lens_test = train_true_len,test_true_len
	train_loader = DataLoader(MyDataset(X_train,y_train,ture_lens_train), batch_size=batch_size, shuffle=True)
	val_loader = DataLoader(MyDataset(X_test,y_test,ture_lens_test), batch_size=batch_size//2, shuffle=False)

	# 模型
	enc_hid = hid_dim
	dec_hid = hid_dim
	attn    = Attention(enc_hid, dec_hid)
	enc     = Encoder(vocab_size, emb_dim, enc_hid, dec_hid, n_layers, dropout)
	dec     = Decoder(vocab_size, emb_dim, enc_hid, dec_hid, dropout, attn)
	model   = Seq2Seq(enc, dec, device).to(device)

	total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
	print(f"模型参数量: {total_params:,}\n")

	optimizer = optim.Adam(model.parameters(), lr=1e-3)
	scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
	criterion = nn.CrossEntropyLoss(ignore_index=pad)

	best_val_loss = float("inf")

	# 训练循环
	for epoch in range(1, n_epochs + 1):
		train_loss = train_epoch(model, train_loader, optimizer, criterion, clip)
		val_loss   = eval_epoch(model, val_loader, criterion)
		scheduler.step(val_loss)

		mark = " ✓ best" if val_loss < best_val_loss else ""
		if val_loss < best_val_loss:
			best_val_loss = val_loss
			torch.save(model.state_dict(), "best_seq2seq.pt")

		print(f"Epoch {epoch:02d}/{n_epochs} | "
			  f"Train Loss: {train_loss:.4f} (PPL: {math.exp(train_loss):6.2f}) | "
			  f"Val Loss: {val_loss:.4f} (PPL: {math.exp(val_loss):6.2f}){mark}")




In [ ]:
# ── 推理演示 ──
print("\n=== 推理示例（加载最优权重）===")
enc_hid = hid_dim
dec_hid = hid_dim
attn    = Attention(enc_hid, dec_hid)
enc     = Encoder(vocab_size, emb_dim, enc_hid, dec_hid, n_layers, dropout)
dec     = Decoder(vocab_size, emb_dim, enc_hid, dec_hid, dropout, attn)
model   = Seq2Seq(enc, dec, device).to(device)
model.load_state_dict(torch.load("best_seq2seq.pt", map_location=device))

test_cases = [
	[3, 1, 4, 1, 5],
	[9, 2, 6],
	[7, 7, 3, 8],
]
for nums in test_cases:
	tok_ids  = [sos] + [n + 3 for n in nums] + [eos]  # +3 因为 0/1/2 是特殊符
	tok_ids  = pad_seq(tok_ids, max_len)
	src_t    = torch.tensor([tok_ids], dtype=torch.long)

	pred_ids = model.translate(src_t)
	src_nums = [idx2tok[i] for i in tok_ids if i not in (pad, sos, eos)]
	pred_nums = [idx2tok[i] for i in pred_ids]

	print(f"  输入: {src_nums}  →  预测: {pred_nums}  "
		  f"(期望: {list(reversed([idx2tok[t+3] for t in nums]))})")

In [72]:
main()

Using device: mps

模型参数量: 5,969,708



KeyboardInterrupt: 